###  Main problem statement: “To determine which client segments generate the highest gross profit while maintaining strong customer satisfaction” (enabling the company to prioritize high-value clients, improve client retention, and support sustainable business growth)​

#### Key points (subproblems)​

#### Profitability and its drivers by client segment: Analyse gross profit and gross margin across client type, industry sector, organisation size and location, while examining hardware, software and manpower costs and service ratings to identify high-value segments and opportunities for cost optimisation and margin improvement.​

In [7]:
import pandas as pd
import plotly.express as px

# ============================================
# Load and Prepare Data
# ============================================
xls = pd.ExcelFile("merged.xlsx")
df_merged = pd.read_excel(xls, xls.sheet_names[0])

# Financial calculations
df_merged["COGS"] = (
    df_merged["HARDWARE"]
    + df_merged["SOFTWARE"]
    + df_merged["MANPOWER"]
)

df_merged["GROSS_PROFIT"] = (
    df_merged["REVENUE"]
    - df_merged["COGS"]
)

df_merged["GROSS_MARGIN"] = (
    df_merged["GROSS_PROFIT"]
    / df_merged["REVENUE"]
) * 100

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

frames = []

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)
        .agg({
            "GROSS_PROFIT": "sum",
            "GROSS_MARGIN": "mean",
            "REVENUE": "sum",
            "NPS RATING": "mean"
        })
        .reset_index()
    )

    grouped = grouped.rename(columns={col: "Segment"})
    grouped["Segmentation"] = label

    # Sort from highest to lowest profit
    grouped = grouped.sort_values(
        "GROSS_PROFIT",
        ascending=False
    )

    frames.append(grouped)

plot_df = pd.concat(frames, ignore_index=True)

# ============================================
# Animated Horizontal Bar Chart
# ============================================

fig = px.bar(

    plot_df,

    x="GROSS_PROFIT",

    y="Segment",

    orientation="h",

    color="GROSS_MARGIN",

    color_continuous_scale="Viridis",

    animation_frame="Segmentation",

    hover_name="Segment",

    hover_data={
        "GROSS_PROFIT": ":,.0f",
        "GROSS_MARGIN": ":.2f",
        "REVENUE": ":,.0f",
        "NPS RATING": ":.2f"
    },

    title="Gross Profit Across Client Segmentations"

)

fig.update_layout(

    template="plotly_white",

    title_x=0.5,

    xaxis_title="Total Gross Profit",

    yaxis_title="Client Segment",

    height=650,

    coloraxis_colorbar=dict(
        title="Gross Margin (%)"
    )

)

fig.show()
fig1 = fig

In [8]:
import plotly.graph_objects as go
import numpy as np

# ============================================
# Create Average Service Rating
# ============================================
service_cols = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT"
]

df_merged["AVG_SERVICE"] = df_merged[service_cols].mean(axis=1)

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

aggregated = {}

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)
        .agg({
            "AVG_SERVICE": "mean",
            "GROSS_MARGIN": "mean",
            "GROSS_PROFIT": "sum",
            "REVENUE": "sum"
        })
        .reset_index()
    )

    aggregated[label] = grouped

# ============================================
# Default View
# ============================================
default = "Industry Sector"

data = aggregated[default]
segment_col = segmentations[default]

# ============================================
# Bubble Size Scaling
# ============================================
bubble_size = (
    data["REVENUE"] / data["REVENUE"].max()
) * 60 + 12

# ============================================
# Figure
# ============================================
fig = go.Figure()

fig.add_trace(

    go.Scatter(

        x=data["AVG_SERVICE"],
        y=data["GROSS_MARGIN"],

        mode="markers+text",

        text=data[segment_col],
        textposition="top center",

        marker=dict(

            size=bubble_size,

            color=data["GROSS_PROFIT"],

            colorscale="Viridis",

            showscale=True,

            colorbar=dict(title="Gross Profit"),

            sizemode="diameter",

            line=dict(width=1)

        ),

        customdata=np.stack(

            (

                data[segment_col],

                data["GROSS_PROFIT"],

                data["REVENUE"]

            ),

            axis=-1

        ),

        hovertemplate=
        "<b>%{customdata[0]}</b><br><br>" +
        "Average Service Rating: %{x:.2f}<br>" +
        "Gross Margin: %{y:.2f}%<br>" +
        "Gross Profit: %{customdata[1]:,.0f}<br>" +
        "Revenue: %{customdata[2]:,.0f}<extra></extra>"

    )

)

# ============================================
# Dropdown
# ============================================
buttons = []

for label, column in segmentations.items():

    temp = aggregated[label]

    bubble_size = (
        temp["REVENUE"] / temp["REVENUE"].max()
    ) * 60 + 12

    buttons.append(

        dict(

            label=label,

            method="update",

            args=[

                {

                    "x":[temp["AVG_SERVICE"]],

                    "y":[temp["GROSS_MARGIN"]],

                    "text":[temp[column]],

                    "customdata":[

                        np.stack(

                            (

                                temp[column],

                                temp["GROSS_PROFIT"],

                                temp["REVENUE"]

                            ),

                            axis=-1

                        )

                    ],

                    "marker":[

                        dict(

                            size=bubble_size,

                            color=temp["GROSS_PROFIT"],

                            colorscale="Viridis",

                            showscale=True,

                            colorbar=dict(title="Gross Profit"),

                            sizemode="diameter",

                            line=dict(width=1)

                        )

                    ]

                },

                {

                    "title":f"Service Quality vs Gross Margin by {label}",

                    "xaxis":{"title":"Average Service Rating"},

                    "yaxis":{"title":"Average Gross Margin (%)"}

                }

            ]

        )

    )

# ============================================
# Layout
# ============================================
fig.update_layout(

    title="Service Quality vs Gross Margin by Industry Sector",

    template="plotly_white",

    xaxis_title="Average Service Rating",

    yaxis_title="Average Gross Margin (%)",

    updatemenus=[

        dict(

            buttons=buttons,

            direction="down",

            x=0.02,

            y=1.18,

            showactive=True

        )

    ]

)

fig.show()
fig2 = fig

In [9]:
import plotly.graph_objects as go
import pandas as pd
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

aggregated = {}

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)[["HARDWARE", "SOFTWARE", "MANPOWER"]]
        .sum()
        .reset_index()
    )

    aggregated[label] = grouped
default = "Industry Sector"

data = aggregated[default]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["HARDWARE"],
        name="Hardware"
    )
)

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["SOFTWARE"],
        name="Software"
    )
)

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["MANPOWER"],
        name="Manpower"
    )
)
buttons = []

for label, column in segmentations.items():

    temp = aggregated[label]

    buttons.append(

        dict(

            label=label,

            method="update",

            args=[

                {
                    "x":[
                        temp[column],
                        temp[column],
                        temp[column]
                    ],

                    "y":[
                        temp["HARDWARE"],
                        temp["SOFTWARE"],
                        temp["MANPOWER"]
                    ]

                },

                {
                    "title":f"Cost Composition by {label}",
                    "xaxis":{"title":label}
                }

            ]

        )

    )
mode_buttons = [

    dict(

        label="Stacked",

        method="relayout",

        args=[{"barmode":"stack"}]

    ),

    dict(

        label="Grouped",

        method="relayout",

        args=[{"barmode":"group"}]

    )

]
fig.update_layout(

    title="Cost Composition by Industry Sector",

    xaxis_title="Industry Sector",

    yaxis_title="Total Cost",

    barmode="stack",

    template="plotly_white",

    updatemenus=[

        dict(

            buttons=buttons,

            direction="down",

            x=0.02,

            y=1.18,

            showactive=True

        ),

        dict(

            buttons=mode_buttons,

            direction="right",

            x=0.55,

            y=1.18,

            showactive=True

        )

    ]

)

fig.show()
fig3 = fig

In [10]:
import dash
from dash import Dash, dcc, html, Input, Output, callback
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
import numpy as np

# --- Sample Dataset Generation ---
np.random.seed(42)
clients = [f"Client {chr(65+i)}" for i in range(12)]
data = []

for client in clients:
    revenue = np.random.uniform(50000, 200000)
    labor_cost = revenue * np.random.uniform(0.3, 0.5)
    overhead_cost = revenue * np.random.uniform(0.1, 0.25)
    material_cost = revenue * np.random.uniform(0.05, 0.15)
    profit = revenue - (labor_cost + overhead_cost + material_cost)
    satisfaction_score = np.random.uniform(6.0, 9.8)
    industry = np.random.choice(["Tech", "Finance", "Healthcare", "Retail"])
    
    data.append({
        "Client": client,
        "Industry": industry,
        "Revenue": round(revenue, 2),
        "Labor Cost": round(labor_cost, 2),
        "Overhead Cost": round(overhead_cost, 2),
        "Material Cost": round(material_cost, 2),
        "Total Profit": round(profit, 2),
        "Profit Margin (%)": round((profit / revenue) * 100, 2),
        "Service Quality Score": round(satisfaction_score, 1)
    })

df = pd.DataFrame(data)

# --- App Setup ---
app = Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

# --- Layout ---
app.layout = dbc.Container(
    [
        # Header
        dbc.Row(
            dbc.Col(
                html.Div(
                    [
                        html.H1("Client Profitability Dashboard", className="text-center my-3 fw-bold"),
                        html.P(
                            "Interactive analysis of client revenue, margins, and service performance.",
                            className="text-center text-muted mb-4"
                        ),
                    ]
                )
            )
        ),
        
        # Interactive Controls Card
        dbc.Card(
            dbc.CardBody(
                dbc.Row(
                    [
                        dbc.Col(
                            [
                                html.Label("Filter by Industry:", className="fw-bold mb-1"),
                                dcc.Dropdown(
                                    id="industry-filter",
                                    options=[{"label": "All Industries", "value": "All"}] + 
                                            [{"label": ind, "value": ind} for ind in sorted(df["Industry"].unique())],
                                    value="All",
                                    clearable=False,
                                    style={"color": "#000"}
                                ),
                            ],
                            xs=12, md=6, className="mb-3 mb-md-0"
                        ),
                        dbc.Col(
                            [
                                html.Label("Minimum Service Quality Score:", className="fw-bold mb-1"),
                                dcc.Slider(
                                    id="quality-slider",
                                    min=5.0,
                                    max=10.0,
                                    step=0.5,
                                    value=5.0,
                                    marks={i: str(i) for i in range(5, 11)},
                                    tooltip={"placement": "bottom", "always_visible": False}
                                ),
                            ],
                            xs=12, md=6
                        ),
                    ]
                )
            ),
            className="mb-4 shadow-sm"
        ),

        # KPI Summary Cards
        dbc.Row(id="kpi-cards", className="mb-4"),

        # Top Section: Overall Profitability Overview
        dbc.Row(
            dbc.Col(
                dbc.Card(
                    [
                        dbc.CardHeader("Profitability Overview by Client", className="fw-bold fs-5"),
                        dbc.CardBody(dcc.Graph(id="fig-profitability")),
                    ],
                    className="shadow-sm mb-4"
                )
            )
        ),

        # Bottom Section: Split View
        dbc.Row(
            [
                dbc.Col(
                    dbc.Card(
                        [
                            dbc.CardHeader("Service Quality vs Profitability", className="fw-bold fs-5"),
                            dbc.CardBody(dcc.Graph(id="fig-quality")),
                        ],
                        className="shadow-sm h-100"
                    ),
                    xs=12, lg=6, className="mb-4 mb-lg-0"
                ),
                dbc.Col(
                    dbc.Card(
                        [
                            dbc.CardHeader("Cost Composition Breakdown", className="fw-bold fs-5"),
                            dbc.CardBody(dcc.Graph(id="fig-costs")),
                        ],
                        className="shadow-sm h-100"
                    ),
                    xs=12, lg=6
                ),
            ]
        ),
    ],
    fluid=True,
    className="py-4 px-3"
)


# --- Callback Logic ---
@callback(
    [
        Output("kpi-cards", "children"),
        Output("fig-profitability", "figure"),
        Output("fig-quality", "figure"),
        Output("fig-costs", "figure"),
    ],
    [
        Input("industry-filter", "value"),
        Input("quality-slider", "value"),
    ]
)
def update_dashboard(selected_industry, min_quality):
    # Filter dataset
    filtered_df = df[df["Service Quality Score"] >= min_quality]
    if selected_industry != "All":
        filtered_df = filtered_df[filtered_df["Industry"] == selected_industry]

    # Handle empty state gracefully
    if filtered_df.empty:
        empty_fig = px.scatter(title="No data available for selected filters.").update_layout(template="plotly_dark")
        return html.Div("No clients match the current filter criteria.", className="text-center text-warning fs-5 fw-bold"), empty_fig, empty_fig, empty_fig

    # 1. KPI Cards Generation
    total_rev = filtered_df["Revenue"].sum()
    total_profit = filtered_df["Total Profit"].sum()
    avg_margin = filtered_df["Profit Margin (%)"].mean()
    avg_quality = filtered_df["Service Quality Score"].mean()

    kpis = [
        ("Total Revenue", f"${total_rev:,.0f}"),
        ("Total Profit", f"${total_profit:,.0f}"),
        ("Avg Profit Margin", f"{avg_margin:.1f}%"),
        ("Avg Quality Score", f"{avg_quality:.1f} / 10"),
    ]

    cards_layout = [
        dbc.Col(
            dbc.Card(
                dbc.CardBody(
                    [
                        html.H6(title, className="card-subtitle text-muted mb-1 fs-7"),
                        html.H3(val, className="card-title fw-bold m-0"),
                    ]
                ),
                className="shadow-sm border-left-accent"
            ),
            xs=6, md=3, className="mb-2 mb-md-0"
        )
        for title, val in kpis
    ]

    # 2. Main Overview Bar Chart
    fig1 = px.bar(
        filtered_df,
        x="Client",
        y="Total Profit",
        color="Profit Margin (%)",
        color_continuous_scale="Viridis",
        hover_data=["Revenue", "Industry"],
        template="plotly_dark"
    )
    fig1.update_layout(margin=dict(l=20, r=20, t=30, b=20))

    # 3. Scatter Plot: Service Quality vs Profitability
    fig2 = px.scatter(
        filtered_df,
        x="Service Quality Score",
        y="Profit Margin (%)",
        size="Revenue",
        color="Industry",
        hover_name="Client",
        size_max=25,
        template="plotly_dark"
    )
    fig2.update_layout(margin=dict(l=20, r=20, t=30, b=20))

    # 4. Stacked Cost Composition Chart
    cost_melted = filtered_df.melt(
        id_vars=["Client"],
        value_vars=["Labor Cost", "Overhead Cost", "Material Cost"],
        var_name="Cost Type",
        value_name="Amount"
    )
    fig3 = px.bar(
        cost_melted,
        x="Client",
        y="Amount",
        color="Cost Type",
        barmode="stack",
        template="plotly_dark"
    )
    fig3.update_layout(margin=dict(l=20, r=20, t=30, b=20))

    return cards_layout, fig1, fig2, fig3


if __name__ == "__main__":
    app.run(debug=True)

### Insight
**Healthcare**, **Info Tech**, and **Manufacturing** are the strongest-performing sectors because they combine:
* High gross profit
* Strong gross margins

This means they are not only generating large amounts of revenue but are also converting revenue into profit efficiently.

---

### Action
The company should:
* Prioritise acquiring and retaining clients in these sectors.
* Allocate more sales and marketing resources to these industries.
* Develop sector-specific offerings for these clients.

### 2. Which sectors have margin improvement opportunities?
*(Using Graph 1 + Graph 3)*

* **Finance**
  * **Margin:** ~43% (good)
  * **Profit:** Relatively low (~4M)
  * *Takeaway:* Finance appears efficient but small in scale.

* **Education**
  * **Margin:** ~38%
  * **Profit:** ~10M
  * *Takeaway:* Moderate profitability with room for improvement.

* **Charity**
  * **Margin:** ~14%
  * **Profit:** Almost negligible
  * *Takeaway:* This sector contributes little profit and has poor margins.

---

### Action

**For low-margin sectors:**
* Review pricing strategy.
* Reduce unnecessary costs.
* Reassess whether these sectors should remain a strategic focus.

**For Charity specifically:**
* Consider whether the sector is strategically valuable.
* If retained, simplify service offerings to improve profitability.

### 3. What drives costs?
*(Using Graph 3)*

Several sectors show that:
* **Manpower** is consistently the largest cost component.
* **Software** is generally the second-largest.
* **Hardware** is often the smallest contributor.

#### Examples
* **Healthcare:** Very high manpower and software costs.
* **Manufacturing:** Large manpower and software expenditure.
* **Info Tech:** Heavy spending across all three categories.

---

### Insight
Profitability is strongly influenced by manpower efficiency. This suggests:
* Staff allocation
* Project productivity
* Resource utilisation

...are major drivers of margin performance.

---

### Action
Possible strategies:
* Improve workforce planning.
* Increase automation.
* Standardise project delivery processes.
* Reduce repetitive manual work.

> **Note:** Even a small reduction in manpower costs could significantly improve margins.

### 4. Does service quality affect profitability?
*(Using Graph 2)*

Graph 2 shows:

| Sector | Service | Margin |
| :--- | :--- | :--- |
| **Finance** | Highest service | Good margin |
| **Transportation** | High service | Highest margin |
| **Info Tech** | High service | Highest margin |
| **Manufacturing** | Good service | Highest margin |

Meanwhile:

| Sector | Service | Margin |
| :--- | :--- | :--- |
| **Charity** | High service | Low margin |

---

### Insight
There appears to be a generally positive relationship between service quality and profitability:
* Sectors with better service ratings often achieve stronger margins.

This suggests that investments in:
* Presales
* Technical expertise
* Delivery quality
* Post-sales support

...may contribute to better financial outcomes.

> **Note:** Charity shows that good service alone does not guarantee profitability, indicating that pricing and cost structures also matter.

### Summary & Conclusion

**Healthcare**, **Info Tech**, and **Manufacturing** are the most attractive client segments because they generate the highest gross profits while maintaining strong gross margins and above-average service ratings.

These sectors should therefore be prioritised for:
* Client acquisition
* Retention initiatives
* Additional investment

---

#### Key Takeaways
* **Cost Optimisation:** Efforts should focus primarily on manpower efficiency, particularly in sectors with high labour costs but weaker margins.
* **Service Quality:** The positive relationship observed between service quality and profitability suggests that maintaining strong customer experience can support both client retention and sustainable business growth.